# Black Friday Sales — EDA (Univariate & Bivariate Analysis)

**Target Variable:** `Purchase`

This notebook is designed to run on **Google Colab**. It covers:
1. Data loading & basic exploration
2. Missing value & unique value checks
3. Data preprocessing (cleaning, renaming, encoding, mapping ranges)
4. Univariate analysis (individual columns, incl. Purchase distribution & outliers)
5. Bivariate analysis (each column vs Purchase)
6. Additional visualizations (correlation, pie chart, etc.)

> Note: `Product_Category_2` and `Product_Category_3` are already **masked** — i.e. the categorical product sub-category values have already been converted to numeric codes by the dataset provider.


## 1. Setup & Data Loading

Run the cell below. If you're on **Google Colab**, it will prompt you to upload `Black_Friday_Sales.csv`. If the file is already in your Colab working directory (or mounted via Google Drive), it will just load it directly.

In [ ]:
# Install/Import required libraries (Colab usually has these pre-installed)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)


In [ ]:
# Load the dataset
# Option A: Running on Google Colab -> upload the file manually
try:
    import google.colab
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

FILE_NAME = 'Black_Friday_Sales.csv'

if IN_COLAB:
    import os
    if not os.path.exists(FILE_NAME):
        print("Please upload 'Black_Friday_Sales.csv'")
        uploaded = files.upload()
        FILE_NAME = list(uploaded.keys())[0]

df = pd.read_csv(FILE_NAME)
print("Shape of dataset:", df.shape)
df.head()


## 2. Basic Statistics of the Dataset

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Statistics for numeric columns only
df.describe()

## 3. Missing Values Check

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
missing_df

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=missing_df.index, y=missing_df['Missing_Percent'])
plt.ylabel('% Missing')
plt.title('Missing Value Percentage by Column')
plt.xticks(rotation=45)
plt.show()

## 4. Unique Values Check

In [ ]:
for col in df.columns:
    print(f"{col:30s} -> {df[col].nunique()} unique values")

In [ ]:
# Look at the unique categories for key columns
for col in ['Gender', 'Age', 'City_Category', 'Stay_In_Current_City_Years',
            'Marital_Status', 'Occupation', 'Product_Category_1']:
    print(f"\n{col}: {sorted(df[col].unique(), key=str)}")

## 5. Data Preprocessing

Steps performed:
- Drop unnecessary fields (`User_ID`, `Product_ID` — pure identifiers, not useful for EDA/modeling)
- Fill missing values in `Product_Category_2` and `Product_Category_3` (0 = "no sub-category")
- Rename a couple of columns for clarity
- Map/encode categorical columns into integers (`Gender`, `Age`, `City_Category`, `Stay_In_Current_City_Years`)


In [ ]:
df_clean = df.copy()

# --- Drop unnecessary identifier columns ---
df_clean.drop(columns=['User_ID', 'Product_ID'], inplace=True)
df_clean.shape

In [ ]:
# --- Fill NaN values in Product_Category_2 and Product_Category_3 ---
# NaN here means the product doesn't have a 2nd/3rd sub-category -> fill with 0
df_clean['Product_Category_2'] = df_clean['Product_Category_2'].fillna(0).astype(int)
df_clean['Product_Category_3'] = df_clean['Product_Category_3'].fillna(0).astype(int)

df_clean[['Product_Category_2', 'Product_Category_3']].isnull().sum()

In [ ]:
# --- Rename columns for clarity ---
df_clean.rename(columns={
    'Stay_In_Current_City_Years': 'Stay_Years',
    'Product_Category_1': 'Product_Cat_1',
    'Product_Category_2': 'Product_Cat_2',
    'Product_Category_3': 'Product_Cat_3'
}, inplace=True)
df_clean.columns.tolist()

In [ ]:
# --- Map Gender into integers ---
gender_map = {'F': 0, 'M': 1}
df_clean['Gender'] = df_clean['Gender'].map(gender_map)

# --- Map Marital_Status is already integer (0 = Single, 1 = Married) ---

# --- Map City_Category into integers ---
city_map = {'A': 0, 'B': 1, 'C': 2}
df_clean['City_Category_Code'] = df_clean['City_Category'].map(city_map)

# --- Clean & map Stay_Years ('4+' -> 4) into integers ---
df_clean['Stay_Years'] = df_clean['Stay_Years'].replace('4+', '4').astype(int)

df_clean.head()

In [ ]:
# --- Map Age ranges into integers (ordinal encoding) ---
age_map = {
    '0-17': 0,
    '18-25': 1,
    '26-35': 2,
    '36-45': 3,
    '46-50': 4,
    '51-55': 5,
    '55+': 6
}
df_clean['Age_Code'] = df_clean['Age'].map(age_map)
df_clean[['Age', 'Age_Code']].drop_duplicates().sort_values('Age_Code')

In [ ]:
# Final check - no missing values, correct dtypes
print(df_clean.isnull().sum())
print()
df_clean.dtypes

In [ ]:
df_clean.head(10)

## 6. Exploratory Data Analysis (EDA)

### 6.1 Purchase Distribution (Target Variable)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))

sns.histplot(df_clean['Purchase'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Purchase Amount')
axes[0].set_xlabel('Purchase')

sns.boxplot(x=df_clean['Purchase'], ax=axes[1], color='orange')
axes[1].set_title('Boxplot of Purchase Amount')

plt.tight_layout()
plt.show()

print(df_clean['Purchase'].describe())

### 6.2 Outlier Detection (IQR method) — Purchase

In [ ]:
Q1 = df_clean['Purchase'].quantile(0.25)
Q3 = df_clean['Purchase'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['Purchase'] < lower_bound) | (df_clean['Purchase'] > upper_bound)]
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
print(f"Number of outliers in Purchase: {len(outliers)} ({len(outliers)/len(df_clean)*100:.2f}% of data)")

### 6.3 Univariate Analysis — Individual Categorical Columns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20,10))

sns.countplot(x='Gender', data=df, ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Gender Distribution')

sns.countplot(x='Age', data=df, ax=axes[0,1], palette='Set2',
              order=['0-17','18-25','26-35','36-45','46-50','51-55','55+'])
axes[0,1].set_title('Age Group Distribution')
axes[0,1].tick_params(axis='x', rotation=45)

sns.countplot(x='City_Category', data=df, ax=axes[0,2], palette='Set2',
              order=sorted(df['City_Category'].unique()))
axes[0,2].set_title('City Category Distribution')

sns.countplot(x='Marital_Status', data=df, ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Marital Status Distribution (0=Single, 1=Married)')

sns.countplot(x='Stay_In_Current_City_Years', data=df, ax=axes[1,1], palette='Set2',
              order=sorted(df['Stay_In_Current_City_Years'].unique()))
axes[1,1].set_title('Stay in Current City (Years) Distribution')

sns.countplot(x='Occupation', data=df, ax=axes[1,2], palette='tab20')
axes[1,2].set_title('Occupation Distribution')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,5))
sns.countplot(x='Product_Category_1', data=df, palette='viridis',
              order=sorted(df['Product_Category_1'].unique()))
plt.title('Product Category 1 Distribution')
plt.show()

## 7. Bivariate Analysis — Each Feature vs Purchase

### 7.1 Age vs Purchase

In [ ]:
age_order = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']

fig, axes = plt.subplots(1, 2, figsize=(16,5))

sns.boxplot(x='Age', y='Purchase', data=df, order=age_order, ax=axes[0], palette='Set3')
axes[0].set_title('Purchase Distribution by Age Group')
axes[0].tick_params(axis='x', rotation=45)

age_purchase = df.groupby('Age')['Purchase'].mean().reindex(age_order)
sns.barplot(x=age_purchase.index, y=age_purchase.values, ax=axes[1], palette='Set3')
axes[1].set_title('Average Purchase by Age Group')
axes[1].set_ylabel('Average Purchase')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 7.2 Gender vs Purchase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(x='Gender', y='Purchase', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Purchase Distribution by Gender')

gender_purchase = df.groupby('Gender')['Purchase'].mean()
sns.barplot(x=gender_purchase.index, y=gender_purchase.values, ax=axes[1], palette='Set2')
axes[1].set_title('Average Purchase by Gender')
axes[1].set_ylabel('Average Purchase')

plt.tight_layout()
plt.show()

### 7.3 Marital Status vs Purchase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(x='Marital_Status', y='Purchase', data=df, ax=axes[0], palette='Set1')
axes[0].set_title('Purchase Distribution by Marital Status')

ms_purchase = df.groupby('Marital_Status')['Purchase'].mean()
sns.barplot(x=ms_purchase.index, y=ms_purchase.values, ax=axes[1], palette='Set1')
axes[1].set_title('Average Purchase by Marital Status (0=Single, 1=Married)')
axes[1].set_ylabel('Average Purchase')

plt.tight_layout()
plt.show()

### 7.4 Occupation vs Purchase

In [ ]:
plt.figure(figsize=(16,6))
occ_purchase = df.groupby('Occupation')['Purchase'].mean().sort_values(ascending=False)
sns.barplot(x=occ_purchase.index, y=occ_purchase.values, order=occ_purchase.index, palette='mako')
plt.title('Average Purchase by Occupation')
plt.xlabel('Occupation')
plt.ylabel('Average Purchase')
plt.show()

### 7.5 Purchase by City Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(x='City_Category', y='Purchase', data=df, order=['A','B','C'], ax=axes[0], palette='cool')
axes[0].set_title('Purchase Distribution by City Category')

city_purchase = df.groupby('City_Category')['Purchase'].mean().reindex(['A','B','C'])
sns.barplot(x=city_purchase.index, y=city_purchase.values, ax=axes[1], palette='cool')
axes[1].set_title('Average Purchase by City Category')
axes[1].set_ylabel('Average Purchase')

plt.tight_layout()
plt.show()

### 7.6 Stay in Current City (Years) vs Purchase

In [ ]:
stay_order = sorted(df['Stay_In_Current_City_Years'].unique())
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(x='Stay_In_Current_City_Years', y='Purchase', data=df, order=stay_order, ax=axes[0], palette='Spectral')
axes[0].set_title('Purchase Distribution by Years Stayed in City')

stay_purchase = df.groupby('Stay_In_Current_City_Years')['Purchase'].mean().reindex(stay_order)
sns.barplot(x=stay_purchase.index, y=stay_purchase.values, ax=axes[1], palette='Spectral')
axes[1].set_title('Average Purchase by Years Stayed in City')
axes[1].set_ylabel('Average Purchase')

plt.tight_layout()
plt.show()

### 7.7 Product_Category_1 vs Purchase

In [ ]:
plt.figure(figsize=(16,6))
pc1_purchase = df.groupby('Product_Category_1')['Purchase'].mean().sort_values(ascending=False)
sns.barplot(x=pc1_purchase.index, y=pc1_purchase.values, order=pc1_purchase.index, palette='crest')
plt.title('Average Purchase by Product Category 1')
plt.xlabel('Product Category 1')
plt.ylabel('Average Purchase')
plt.show()

### 7.8 Product_Category_2 vs Purchase

In [ ]:
plt.figure(figsize=(16,6))
pc2 = df.dropna(subset=['Product_Category_2'])
pc2_purchase = pc2.groupby('Product_Category_2')['Purchase'].mean().sort_values(ascending=False)
sns.barplot(x=pc2_purchase.index, y=pc2_purchase.values, order=pc2_purchase.index, palette='flare')
plt.title('Average Purchase by Product Category 2 (excluding missing)')
plt.xlabel('Product Category 2')
plt.ylabel('Average Purchase')
plt.show()

### 7.9 Product_Category_3 vs Purchase

In [ ]:
plt.figure(figsize=(16,6))
pc3 = df.dropna(subset=['Product_Category_3'])
pc3_purchase = pc3.groupby('Product_Category_3')['Purchase'].mean().sort_values(ascending=False)
sns.barplot(x=pc3_purchase.index, y=pc3_purchase.values, order=pc3_purchase.index, palette='rocket')
plt.title('Average Purchase by Product Category 3 (excluding missing)')
plt.xlabel('Product Category 3')
plt.ylabel('Average Purchase')
plt.show()

## 8. Additional Visualizations

### 8.1 City Category — Pie Chart

In [ ]:
city_counts = df['City_Category'].value_counts().reindex(['A','B','C'])

plt.figure(figsize=(7,7))
plt.pie(city_counts.values, labels=city_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('pastel'), startangle=90,
        wedgeprops={'edgecolor':'white'})
plt.title('City Category Distribution')
plt.show()

### 8.2 Gender — Pie Chart

In [ ]:
gender_counts = df['Gender'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(gender_counts.values, labels=['Male' if g=='M' else 'Female' for g in gender_counts.index],
        autopct='%1.1f%%', colors=['#66b3ff','#ff9999'], startangle=90,
        wedgeprops={'edgecolor':'white'})
plt.title('Gender Distribution')
plt.show()

### 8.3 Correlation Heatmap (numeric / encoded columns)

In [ ]:
corr_cols = ['Gender', 'Age_Code', 'Occupation', 'City_Category_Code', 'Stay_Years',
             'Marital_Status', 'Product_Cat_1', 'Product_Cat_2', 'Product_Cat_3', 'Purchase']

plt.figure(figsize=(10,8))
sns.heatmap(df_clean[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

### 8.4 Gender vs Age — Purchase (Multi-variable view)

In [ ]:
plt.figure(figsize=(14,6))
sns.barplot(x='Age', y='Purchase', hue='Gender', data=df, order=age_order, palette='Set2')
plt.title('Average Purchase by Age Group and Gender')
plt.ylabel('Average Purchase')
plt.xticks(rotation=45)
plt.legend(title='Gender')
plt.show()

### 8.5 City Category vs Age — Purchase (Multi-variable view)

In [ ]:
plt.figure(figsize=(14,6))
sns.barplot(x='Age', y='Purchase', hue='City_Category', data=df, order=age_order, palette='cool')
plt.title('Average Purchase by Age Group and City Category')
plt.ylabel('Average Purchase')
plt.xticks(rotation=45)
plt.legend(title='City Category')
plt.show()

### 8.6 Count of Purchases per City Category & Gender

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(x='City_Category', hue='Gender', data=df, order=['A','B','C'], palette='Set2')
plt.title('Count of Purchases by City Category and Gender')
plt.show()

### 8.7 Top 10 Most Frequently Purchased Products (by count)

In [ ]:
top_products = df['Product_ID'].value_counts().head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_products.values, y=top_products.index, palette='magma')
plt.title('Top 10 Most Frequently Purchased Products')
plt.xlabel('Number of Purchases')
plt.ylabel('Product ID')
plt.show()

## 9. Key Observations (fill in / adjust after reviewing your plots)

- Purchase amounts are right-skewed with a cluster of high-value outliers (see IQR outlier count above).
- Males made noticeably more purchases than females (check the Gender pie chart / countplot), though average purchase per transaction differs — check `Gender vs Purchase` bar chart for the actual direction.
- The `26-35` age group is typically the most active shopper segment for this dataset.
- City Category `B` usually has the highest transaction volume, but check `City vs Purchase` boxplot for which city has the highest **average** purchase.
- `Product_Category_1` values like 1, 5, and 8 tend to dominate in volume, while certain other categories drive higher average purchase value.
- `Occupation` shows some variation in average purchase but not a dramatic spread across groups.

Feel free to extend this notebook with a train/test split and a regression model (e.g. Linear Regression, Random Forest, XGBoost) to predict `Purchase` using the encoded features in `df_clean`.
